# Gemma 4 12B: does steering built from typed remarks also work on spoken remarks?

**On the Colab website:** put `gemma_speech_bundle.tar.gz` in Google Drive at `My Drive/idea3_gemma_speech/`. File > Upload notebook (this file). Runtime > Change runtime type > **A100** (or H100 / G4). Then **Runtime > Run all**. Allow Drive access when asked.

**In VS Code (Colab extension):** Select Kernel > Colab > New Colab Server > GPU > A100. Right-click the bundle > Upload to Colab. Command Palette > Colab: Mount Google Drive to Server. Run all cells.

**What it does:** checks the model, checks it can hear the spoken remark, measures bias in typed and spoken remarks, builds the appearance direction from typed remarks only, picks the steering setting on typed remarks, then applies it to spoken remarks, compared with a direction built from speech (the ceiling) and with control directions (weather mood, shuffled labels, random) and a written rule. An exploratory face arm shows a second image with a face rated more or less trustworthy. Levels: L1 (main) and L2.

**Cost:** roughly 6 to 9 hours on an A100 (about 40 to 60 units). If Colab disconnects, run all cells again: it continues where it stopped. It stops by itself if a check fails. **The last cells make one zip to send back, then release the GPU.**

Nothing here can run the test scenes.

In [ ]:
import os, subprocess
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive; drive.mount('/content/drive')   # Colab website; in VS Code use the Command Palette instead
    except Exception as ex:
        print('Drive not mounted here:', ex)
print(subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; df -h /content | tail -1; python3 --version', shell=True, capture_output=True, text=True).stdout)
assert os.path.exists('/content/drive/MyDrive'), 'Google Drive is NOT mounted. Without it the results are lost when the GPU is released. Mount Drive (website: allow access when asked; VS Code: Command Palette > Colab: Mount Google Drive to Server), then Run all again.'
print('Drive mounted: yes')

In [ ]:
%%writefile /content/gs_colab.py
"""Colab helper for the Gemma 4 12B speech-transfer test (src/gemma_speech.py). Plain Python. The experiment runs in its own uv
environment (torch 2.9.1 + transformers 5.22.2 + pillow 12.3.0). Resumable: results are copied to Drive every minute and restored on
start. The test split is not in this notebook."""
import json, os, re, shutil, subprocess, threading, time
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/idea3_gemma_speech') if Path('/content/drive/MyDrive').exists() else Path('/content/gs_state')
STATE, ROOT, CODE, VENV = DRIVE / 'results', Path('/content/gs'), Path('/content/gs/final_omni'), Path('/content/gsenv')
PY, HF, P777 = VENV / 'bin/python', Path('/content/hf'), Path('/content/gs/p777')
BUNDLE = 'gemma_speech_bundle.tar.gz'
SCRIPT, SUB = 'src/gemma_speech.py', 'results/gemma'   # stage script and its results folder (the pilot runner changes both)
MODEL_ID, REV = 'google/gemma-4-12B-it', '43c9d014b231474fa7212dbb48a762566c2b255d'   # a runner may switch the model (qwen_run.py)


def sh(cmd, **k):
    print('$', cmd if isinstance(cmd, str) else ' '.join(map(str, cmd)), flush=True); subprocess.run(cmd, shell=isinstance(cmd, str), check=True, **k)


def env(stop_at=None):
    e = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME', 'MPLBACKEND', 'ALLOW_TEST')}
    e.update(P777=str(P777), DATA_DIR=str(P777 / 'data/egonormia'), REF_DIR=str(P777 / 'data/ref'), HF_HOME=str(HF), HF_HUB_OFFLINE='1',
             TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1', MPLBACKEND='Agg', PATH=f'{VENV}/bin:' + os.environ['PATH'])
    if stop_at:
        e['STOP_AT'] = str(stop_at)
    return e


def gpu_check(min_gb=38):
    out = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits', shell=True, capture_output=True, text=True).stdout.strip()
    print('GPU:', out or 'NONE')
    assert out and ',' in out, 'No GPU found. Choose a GPU machine (A100, L40S, G4 or g7e) and connect again.'
    assert int(out.splitlines()[0].split(',')[1]) > min_gb * 1000, f'GPU is smaller than this model needs (about {min_gb + 2} GB or more); disconnect so it does not cost money.'


def copy_new(src, dst):
    n = 0
    for p in src.rglob('*') if src.exists() else []:
        if p.is_dir() or p.suffix in ('.tmp', '.part'):
            continue
        q = dst / p.relative_to(src); s = p.stat()
        if q.exists() and q.stat().st_size == s.st_size and abs(q.stat().st_mtime - s.st_mtime) < 2:
            continue
        q.parent.mkdir(parents=True, exist_ok=True); t = q.with_name(q.name + '.part'); shutil.copy2(p, t); os.replace(t, q); os.utime(q, (s.st_mtime, s.st_mtime)); n += 1
    return n


def sync():
    return copy_new(CODE / SUB, STATE)


def setup():
    if not str(DRIVE).startswith('/content/drive'):
        print('WARNING: Drive is not mounted: progress is LOST on disconnect. Command Palette > "Colab: Mount Google Drive to Server", then run from the top.')
    DRIVE.mkdir(parents=True, exist_ok=True)
    if not (ROOT / '.unpacked').exists():
        found = [c for c in (Path('/content') / BUNDLE, DRIVE / BUNDLE) if c.exists()]
        assert found, f'Upload {BUNDLE} first (right-click it in VS Code > "Upload to Colab").'
        ROOT.mkdir(parents=True, exist_ok=True); sh(['tar', 'xzf', str(found[0]), '-C', str(ROOT)])
        if not (DRIVE / BUNDLE).exists():
            shutil.copy(found[0], DRIVE / BUNDLE)
        if STATE.exists():
            print('restored from Drive:', copy_new(STATE, CODE / SUB), 'files')
        (ROOT / '.unpacked').touch()
    if not (VENV / '.ok').exists():
        sh('pip -q install uv'); sh(f'uv venv -q --clear --python 3.12 {VENV}')
        sh(f'uv pip install -q --python {PY} --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu128 torch==2.9.1+cu128 torchvision==0.24.1+cu128 transformers==5.22.2 "accelerate>=1.0" pillow==12.3.0 soundfile pyyaml matplotlib numpy requests huggingface_hub hf_xet')
        (VENV / '.ok').touch()
    e = env()
    if not (P777 / 'data/.images_ok').exists():
        sh([str(PY), '-I', 'src/download_data.py', '--workers', '32'], cwd=CODE, env=e)
        rep = json.loads((P777 / 'data/egonormia/verify_report.json').read_text())
        assert rep['raw_sha_ok'] == 1853 and rep['preview_sha_ok'] == 1853, rep
        (P777 / 'data/.images_ok').touch()
    if not (P777 / 'data/omi/.ok').exists():
        # One Million Impressions faces (public, CC BY-NC-SA 4.0), the midterm's pinned commit; used by the exploratory face arm
        sh(f"git clone -q https://github.com/jcpeterson/omi.git {P777 / 'data/omi'} && cd {P777 / 'data/omi'} && git checkout -q 53bb3c13113afc3ae67aa6ede4ac9dfe33f306af && ls images | wc -l")
        (P777 / 'data/omi/.ok').touch()
    wmark = HF / ('.weights_ok' if MODEL_ID == 'google/gemma-4-12B-it' else f'.weights_ok_{REV[:8]}')
    if not wmark.exists():
        e2 = dict(env(), HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0')
        print(f'Downloading {MODEL_ID} (Gemma about 24 GB, Qwen3-Omni about 66 GB)', flush=True)
        sh([str(PY), '-c', f"from huggingface_hub import snapshot_download as s; print(s('{MODEL_ID}', revision='{REV}', max_workers=16))"], env=e2); wmark.touch()
    out = subprocess.run([str(PY), '-c', 'import torch, transformers, PIL; print(torch.__version__, transformers.__version__, PIL.__version__, torch.cuda.get_device_name(0), torch.cuda.is_bf16_supported())'], env=env(), capture_output=True, text=True)
    print('environment:', out.stdout.strip() or out.stderr[-600:])
    (CODE / SUB).mkdir(parents=True, exist_ok=True)
    (CODE / SUB / 'environment.txt').write_text(out.stdout + out.stderr[-2000:] + subprocess.run([str(PY), '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)
    assert out.returncode == 0; print('SETUP DONE')


def stage(name, end):
    log = CODE / SUB / 'colab_log.txt'; log.parent.mkdir(parents=True, exist_ok=True); print(f'=== {name}', time.strftime('%H:%M'), flush=True)
    p = subprocess.Popen([str(PY), SCRIPT, *name.split()], cwd=CODE, env=env(end), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    with open(log, 'a') as fh:
        for line in p.stdout:
            fh.write(line); fh.flush()
            if not re.search(r'it/s|Loading checkpoint|Loading weights|FutureWarning|UserWarning|^\s*$', line):
                print(line.rstrip()[:420], flush=True)
    p.wait(); print(f'=== {name} exit {p.returncode}', time.strftime('%H:%M'), flush=True); return p.returncode


def run(stages, hours=8.0):
    """Runs the stages in order, resumably, until done or the time limit. Stops at the first stage that fails or stops itself."""
    end = time.time() + hours * 3600; stop_sync = threading.Event()
    def loop():
        while not stop_sync.wait(60):
            try: sync()
            except Exception as ex: print('sync problem (will retry):', ex, flush=True)
    threading.Thread(target=loop, daemon=True).start()
    try:
        for s in stages:
            code = stage(s, end)
            if (CODE / SUB / 'STOP.md').exists():
                print('STOPPED BY A GATE:', (CODE / SUB / 'STOP.md').read_text()); break
            if code == 75 or time.time() > end - 120:
                print('time limit reached: Runtime > Run all again to continue (it resumes)'); break
            if code:
                print('stage failed; send this output to Claude'); break
        else:
            print('ALL STAGES IN THIS CELL DONE')
    finally:
        stop_sync.set(); time.sleep(1)
        for _ in range(3):
            if sync() == 0: break
        print('copied to Drive:', STATE)


def release():
    print('Releasing the GPU'); time.sleep(30)
    try:
        from google.colab import drive; drive.flush_and_unmount()   # make sure everything reached Drive first
    except Exception as ex:
        print('Drive flush skipped:', ex)
    try:
        from google.colab import runtime; runtime.unassign()
    except Exception as ex:
        print('Could not release automatically: Command Palette > "Colab: Remove Server" NOW.', ex)


In [ ]:
import sys; sys.path.insert(0, '/content')
import gs_colab as q
q.gpu_check()   # stops here if the GPU is too small
q.setup()       # about 15 to 25 minutes the first time (packages, images, 24 GB model)

## Part A: checks and bias in typed and spoken remarks (no steering). About 1.5 to 2.5 hours
1. model check (letters, greedy answer, steering hook)  2. can it hear the remark (90%)  3. unsteered answers on 300 training scenes  4. bias gate per level

In [ ]:
q.run(['check', 'perc', 'g2', 'gate'], hours=5)

## Part B: steering built from typed remarks, applied to spoken remarks, plus the face arm. About 4 to 6 hours
Runs only if Part A passed. Levels that failed the gate are skipped automatically.

In [ ]:
G = '/content/gs/final_omni/results/gemma/'
if os.path.exists(G + 'STOP.md'):
    print('Stopped in Part A:', open(G + 'STOP.md').read())
elif not os.path.exists(G + 'gate.json'):
    print('Part A is not finished (time limit or disconnect): run all cells again; it continues where it stopped.')
else:
    q.run(['directions', 'sweep', 'select', 'transfer', 'report', 'faces', 'report'], hours=9)   # report runs before and after the face arm

In [ ]:
# send-back file: answer rows, directions, summaries (no model, no images, no per-scene activations). Saved to Drive and /content
import shutil, time
q.sync()
import zipfile
from pathlib import Path
src = Path('/content/gs/final_omni/results/gemma'); z = f'/content/gemma_results_{time.strftime("%m%d_%H%M")}.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as f:
    for p in sorted(src.rglob('*')):
        if p.is_file() and 'acts' not in p.parts:   # per-scene activation files stay on Drive (large)
            f.write(p, p.relative_to(src))
if os.path.exists('/content/drive/MyDrive'): shutil.copy(z, '/content/drive/MyDrive/idea3_gemma_speech/')
print('send this file:', z, round(os.path.getsize(z) / 1e6, 1), 'MB')

In [ ]:
# release the GPU so it stops using units (only once the zip is safely on Drive)
if os.path.exists(f'/content/drive/MyDrive/idea3_gemma_speech/{os.path.basename(z)}'):
    q.release()
else:
    print('The zip is NOT on Drive, so the GPU is not released automatically. Download', z, 'from the Files panel first, then Runtime > Disconnect and delete runtime.')